In [ ]:
!pip install tslearn sktime scikit-learn mantis-tsfm

In [3]:
import numpy as np
from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder

from sktime.classification.kernel_based import RocketClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

import time
import torch
from mantis.architecture import Mantis8M
from mantis.trainer import MantisTrainer
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression

In [4]:
print("--- Chargement du dataset LSST ---")
ds = UCR_UEA_datasets()
X_train_raw, y_train_raw, X_test_raw, y_test_raw = ds.load_dataset("LSST")

# 1. Ajustement des dimensions (N, Timesteps, Channels) -> (N, Channels, Timesteps)
X_train = X_train_raw.swapaxes(1, 2)
X_test = X_test_raw.swapaxes(1, 2)

# 2. Encodage des labels textuels en entiers continus (requis par les réseaux de neurones)
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

print(f"Train shape : {X_train.shape}")
print(f"Test shape  : {X_test.shape}")

--- Chargement du dataset LSST ---
Train shape : (2459, 6, 36)
Test shape  : (2466, 6, 36)


## Rocket

In [5]:
print("--- Baseline : Rocket ---")
clf_rocket = RocketClassifier(num_kernels=10000, random_state=42)

# Mesure du temps d'entraînement
start_train = time.time()
clf_rocket.fit(X_train, y_train)
end_train = time.time()

# Mesure du temps d'inférence
start_test = time.time()
preds_rocket = clf_rocket.predict(X_test)
end_test = time.time()

print(f"Training Time : {end_train - start_train:.2f} sec")
print(f"Inference Time : {end_test - start_test:.2f} sec")

--- Baseline : Rocket ---
Training Time : 181.97 sec
Inference Time : 54.98 sec


In [9]:
acc_rocket = accuracy_score(y_test, preds_rocket)
f1_rocket = f1_score(y_test, preds_rocket, average='weighted')

print(f"Accuracy ROCKET : {acc_rocket * 100:.2f}%")
print(f"F1-Score (Weighted) ROCKET  : {f1_rocket * 100:.2f}%")
print("\n--- Report ---")
print(classification_report(y_test, preds_rocket))

Accuracy ROCKET : 64.44%
F1-Score (Weighted) ROCKET  : 60.28%

--- Report ---
              precision    recall  f1-score   support

           0       0.72      0.47      0.57       124
           1       0.80      0.91      0.85       270
           2       0.48      0.47      0.48       382
           3       0.00      0.00      0.00        63
           4       0.00      0.00      0.00         7
           5       1.00      0.09      0.16        35
           6       0.35      0.15      0.21       153
           7       1.00      0.04      0.08        24
           8       0.73      0.87      0.79       313
           9       0.56      0.07      0.13        68
          10       0.90      0.94      0.92       121
          11       0.61      0.84      0.70       777
          12       0.70      0.42      0.52        77
          13       0.67      0.12      0.20        52

    accuracy                           0.64      2466
   macro avg       0.61      0.38      0.40      2466
we

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [6]:
# Détection automatique du GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilisation de l'accélérateur : {device}")

# Chargement du Foundation Model pré-entraîné
network = Mantis8M(device=device)
network = network.from_pretrained("paris-noah/Mantis-8M")

Utilisation de l'accélérateur : cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/32.5M [00:00<?, ?B/s]

## Resize of the datasets

In [7]:
# 1. Fonction pour redimensionner à 512 (recommandation officielle MANTIS)
def resize_for_mantis(X, target_length=512):
    # X est de forme (N, Channels, Timesteps)
    X_tensor = torch.tensor(X, dtype=torch.float)
    # L'interpolation linéaire adapte la série à la nouvelle longueur
    X_scaled = F.interpolate(X_tensor, size=target_length, mode='linear', align_corners=False)
    return X_scaled.numpy()

print("--- Redimensionnement des données pour MANTIS ---")
X_train_mantis = resize_for_mantis(X_train)
X_test_mantis = resize_for_mantis(X_test)
print(f"Nouvelle dimension Train : {X_train_mantis.shape}")

--- Redimensionnement des données pour MANTIS ---
Nouvelle dimension Train : (2459, 6, 512)


## Extraction des features + Logistic regression

In [11]:
# MANTIS : Extraction de features + Logistic Regression
model_mantis_feat = MantisTrainer(network=network, device=device)

# Temps d'entraînement : Extraction Train + Fit LR
start_train = time.time()
Z_train_mantis = model_mantis_feat.transform(X_train_mantis)
clf_lr = LogisticRegression(max_iter=1000, random_state=42)
clf_lr.fit(Z_train_mantis, y_train)
end_train = time.time()

# Temps d'inférence : Extraction Test + Score LR
start_test = time.time()
Z_test_mantis = model_mantis_feat.transform(X_test_mantis)
preds_mantis_feat = clf_lr.predict(Z_test_mantis)
end_test = time.time()
print(f"Training time : {end_train - start_train:.2f} secs")
print(f"Inference time : {end_test - start_test:.2f} sec")

Training time : 28.96 secs
Inference time : 4.52 sec


In [13]:
time_train_feat = end_train - start_train
time_test_feat = end_test - start_test
acc_mantis_feat = accuracy_score(y_test, preds_mantis_feat)
f1_mantis_feat = f1_score(y_test, preds_mantis_feat, average='weighted')
report_mantis_feat = classification_report(y_test, preds_mantis_feat)
print("=== RÉSULTATS : MANTIS (Features + LR) ===")
print(f"Précision (Accuracy) : {acc_mantis_feat * 100:.2f}%")
print(f"F1-Score (Weighted)  : {f1_mantis_feat * 100:.2f}%")
print(f"Training time        : {time_train_feat:.2f} sec")
print(f"Inference time       : {time_test_feat:.2f} sec")
print("-" * 40)
print("Classification Report :")
print(report_mantis_feat)

=== RÉSULTATS : MANTIS (Features + LR) ===
Précision (Accuracy) : 60.26%
F1-Score (Weighted)  : 60.14%
Training time        : 28.96 sec
Inference time       : 4.52 sec
----------------------------------------
Classification Report :
              precision    recall  f1-score   support

           0       0.47      0.47      0.47       124
           1       0.95      0.91      0.93       270
           2       0.42      0.47      0.45       382
           3       0.04      0.03      0.03        63
           4       1.00      0.86      0.92         7
           5       0.60      0.34      0.44        35
           6       0.25      0.27      0.26       153
           7       0.07      0.04      0.05        24
           8       0.80      0.76      0.78       313
           9       0.08      0.07      0.08        68
          10       0.96      0.94      0.95       121
          11       0.62      0.64      0.63       777
          12       0.89      0.92      0.90        77
          

## MANTIS : Adaptation 'Head' (Linear Probing)

In [14]:

model_mantis_head = MantisTrainer(network=network, device=device)

# Temps d'entraînement
start_train = time.time()
model_mantis_head.fit(
    X_train_mantis,
    y_train,
    num_epochs=20,
    fine_tuning_type="head"
)
end_train = time.time()

# Temps d'inférence
start_test = time.time()
preds_head = model_mantis_head.predict(X_test_mantis)
end_test = time.time()


Epoch 19: Train Loss 1.1491: 100%|██████████| 20/20 [00:00<00:00, 21.48it/s]


In [18]:
time_train_head = end_train - start_train
time_test_head = end_test - start_test
acc_head = accuracy_score(y_test, preds_head)
f1_head = f1_score(y_test, preds_head, average='weighted', zero_division=0)
report_head = classification_report(y_test, preds_head, zero_division=0)

print("=== RÉSULTATS : MANTIS (Adaptation 'Head' / Linear Probing) ===")
print(f"Précision (Accuracy) : {acc_head * 100:.2f}%")
print(f"F1-Score (Weighted)  : {f1_head * 100:.2f}%")
print(f"Training Time        : {time_train_head:.2f} sec")
print(f"Inference Time       : {time_test_head:.2f} sec")
print("-" * 40)
print("Classification Report :")
print(report_head)

=== RÉSULTATS : MANTIS (Adaptation 'Head' / Linear Probing) ===
Précision (Accuracy) : 60.50%
F1-Score (Weighted)  : 54.34%
Training Time        : 9.70 sec
Inference Time       : 3.98 sec
----------------------------------------
Classification Report :
              precision    recall  f1-score   support

           0       0.63      0.18      0.28       124
           1       0.90      0.88      0.89       270
           2       0.46      0.33      0.39       382
           3       0.00      0.00      0.00        63
           4       0.00      0.00      0.00         7
           5       0.50      0.09      0.15        35
           6       0.09      0.01      0.01       153
           7       0.00      0.00      0.00        24
           8       0.71      0.78      0.75       313
           9       0.00      0.00      0.00        68
          10       0.91      0.88      0.90       121
          11       0.51      0.88      0.65       777
          12       0.78      0.90      0.84 

## MANTIS : Adaptation 'Full' (Full Fine-Tuning)

In [19]:
model_mantis_full = MantisTrainer(network=network, device=device)

# Temps d'entraînement
start_train = time.time()
model_mantis_full.fit(
    X_train_mantis,
    y_train,
    num_epochs=20,
    fine_tuning_type="full"
)
end_train = time.time()

# Temps d'inférence
start_test = time.time()
preds_full = model_mantis_full.predict(X_test_mantis)
end_test = time.time()

print(f"Training Time : {end_train - start_train:.2f} sec")
print(f"Inference Time : {end_test - start_test:.2f} sec")

Epoch 19: Train Loss 0.4511: 100%|██████████| 20/20 [03:56<00:00, 11.85s/it]


Training Time : 236.98 sec
Inference Time : 4.25 sec


In [20]:
# Calcul et stockage des variables (Temps et Scores)
time_train_full = end_train - start_train
time_test_full = end_test - start_test
acc_full = accuracy_score(y_test, preds_full)
f1_full = f1_score(y_test, preds_full, average='weighted')
report_full = classification_report(y_test, preds_full)

print("=== RÉSULTATS : MANTIS (Full Fine-Tuning) ===")
print(f"Précision (Accuracy) : {acc_full * 100:.2f}%")
print(f"F1-Score (Weighted)  : {f1_full * 100:.2f}%")
print(f"Training Time        : {time_train_full:.2f} sec")
print(f"Inference Time       : {time_test_full:.2f} sec")
print("-" * 40)
print("Classification Report :")
print(report_full)

=== RÉSULTATS : MANTIS (Full Fine-Tuning) ===
Précision (Accuracy) : 70.92%
F1-Score (Weighted)  : 68.40%
Training Time        : 236.98 sec
Inference Time       : 4.25 sec
----------------------------------------
Classification Report :
              precision    recall  f1-score   support

           0       0.76      0.50      0.60       124
           1       0.99      0.96      0.97       270
           2       0.48      0.62      0.54       382
           3       0.00      0.00      0.00        63
           4       1.00      0.57      0.73         7
           5       0.74      0.57      0.65        35
           6       0.43      0.23      0.30       153
           7       0.50      0.08      0.14        24
           8       0.80      0.93      0.86       313
           9       0.46      0.09      0.15        68
          10       0.99      0.98      0.99       121
          11       0.69      0.81      0.75       777
          12       0.90      0.96      0.93        77
      